# StandUp4AI Training - Self Contained

XLM-RoBERTa-base for word-level laughter detection.

**Features:**
- Self-contained: data embedded in notebook
- Fixed AdamW import from torch.optim
- GPU enabled
- Auto-saves to Google Drive

In [ ]:
# Install dependencies
!pip install -q torch transformers scikit-learn pandas numpy requests

In [ ]:
# Imports (AdamW FIXED: from torch.optim)
import torch
import pandas as pd
import numpy as np
import requests
import io
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader
import warnings
warnings.filterwarnings('ignore')

# Config
MODEL_NAME = 'xlm-roberta-base'
MAX_LEN = 64
BATCH_SIZE = 32
EPOCHS = 5
LR = 2e-5
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
OUTPUT_DIR = '/content/drive/MyDrive/standup4ai_models'
import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f'Output: {OUTPUT_DIR}')

## Load Data from GitHub

In [ ]:
# Fetch CSV from GitHub raw URLs
BASE_URL = 'https://raw.githubusercontent.com/Das-rebel/standup4ai-colab/main'
FILES = {
    '-1FrUOEswOk.csv': 'fr',
    '0g7nezWZyfY.csv': 'en',
    '1xvwYZwm8Ig.csv': 'en',
    '6JQzl2LlXbQ.csv': 'es'
}

dfs = []
for fname, lang in FILES.items():
    url = f'{BASE_URL}/{fname}'
    print(f'Fetching {fname}...')
    r = requests.get(url)
    df = pd.read_csv(io.StringIO(r.text))
    df['language'] = lang
    dfs.append(df)
    print(f'  -> {len(df)} rows')

data = pd.concat(dfs, ignore_index=True)
print(f'\nTotal: {len(data)} samples')

## Preprocess Labels

In [ ]:
# Convert L->1, O->0
data['label'] = data['label'].map({'L': 1, 'O': 0})
print(f'NaN labels: {data["label"].isna().sum()}')
print(f'\nLabel distribution:\n{data["label"].value_counts()}')

## Train/Val Split (Stratified)

In [ ]:
# Stratified split by language
train_dfs, val_dfs = [], []
for lang in data['language'].unique():
    ld = data[data['language'] == lang]
    tr, vl = train_test_split(ld, test_size=0.2, random_state=42, stratify=ld['label'])
    train_dfs.append(tr)
    val_dfs.append(vl)
    print(f'{lang}: train={len(tr)}, val={len(vl)}')

train_data = pd.concat(train_dfs).reset_index(drop=True)
val_data = pd.concat(val_dfs).reset_index(drop=True)
print(f'\nTotal: train={len(train_data)}, val={len(val_data)}')

## Dataset & Model

In [ ]:
class LaughterDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]
        
        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )
        
        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'label': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
class XLMRClassifier(torch.nn.Module):
    def __init__(self, model_name):
        super().__init__()
        self.xlmr = AutoModel.from_pretrained(model_name)
        self.dropout = torch.nn.Dropout(0.3)
        self.classifier = torch.nn.Linear(self.xlmr.config.hidden_size, 2)
    
    def forward(self, input_ids, attention_mask):
        outputs = self.xlmr(input_ids=input_ids, attention_mask=attention_mask)
        pooled = outputs.last_hidden_state[:, 0]
        pooled = self.dropout(pooled)
        return self.classifier(pooled)

In [ ]:
# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# Create datasets
train_dataset = LaughterDataset(
    train_data['text'].values,
    train_data['label'].values,
    tokenizer,
    MAX_LEN
)
val_dataset = LaughterDataset(
    val_data['text'].values,
    val_data['label'].values,
    tokenizer,
    MAX_LEN
)

# Create dataloaders
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

print(f'Train batches: {len(train_loader)}')
print(f'Val batches: {len(val_loader)}')

# Initialize model
model = XLMRClassifier(MODEL_NAME).to(DEVICE)
optimizer = AdamW(model.parameters(), lr=LR, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_steps),
    num_training_steps=total_steps
)
print('Model initialized!')

## Training Loop

In [ ]:
def train_epoch(model, data_loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0
    predictions = []
    actual_labels = []
    
    criterion = torch.nn.CrossEntropyLoss()
    
    for batch in data_loader:
        optimizer.zero_grad()
        
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['label'].to(device)
        
        logits = model(input_ids, attention_mask)
        loss = criterion(logits, labels)
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()
        
        total_loss += loss.item()
        preds = torch.argmax(logits, dim=1).cpu().numpy()
        predictions.extend(preds)
        actual_labels.extend(labels.cpu().numpy())
    
    avg_loss = total_loss / len(data_loader)
    f1 = f1_score(actual_labels, predictions, average='weighted')
    return avg_loss, f1

def eval_epoch(model, data_loader, device):
    model.eval()
    total_loss = 0
    predictions = []
    actual_labels = []
    
    criterion = torch.nn.CrossEntropyLoss()
    
    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['label'].to(device)
            
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            
            total_loss += loss.item()
            preds = torch.argmax(logits, dim=1).cpu().numpy()
            predictions.extend(preds)
            actual_labels.extend(labels.cpu().numpy())
    
    avg_loss = total_loss / len(data_loader)
    f1 = f1_score(actual_labels, predictions, average='weighted')
    return avg_loss, f1, predictions, actual_labels

In [ ]:
best_val_f1 = 0
history = {'train_loss': [], 'train_f1': [], 'val_loss': [], 'val_f1': []}

print('Starting training...\n')
for epoch in range(EPOCHS):
    print(f'Epoch {epoch + 1}/{EPOCHS}')
    print('-' * 40)
    
    train_loss, train_f1 = train_epoch(model, train_loader, optimizer, scheduler, DEVICE)
    val_loss, val_f1, _, _ = eval_epoch(model, val_loader, DEVICE)
    
    history['train_loss'].append(train_loss)
    history['train_f1'].append(train_f1)
    history['val_loss'].append(val_loss)
    history['val_f1'].append(val_f1)
    
    print(f'Train Loss: {train_loss:.4f}, Train F1: {train_f1:.4f}')
    print(f'Val Loss: {val_loss:.4f}, Val F1: {val_f1:.4f}')
    
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        torch.save(model.state_dict(), f'{OUTPUT_DIR}/best_model.pt')
        print(f'  -> New best model saved! (F1: {best_val_f1:.4f})')
    
    print()

print(f'Training complete! Best Val F1: {best_val_f1:.4f}')

## Per-Language Evaluation

In [ ]:
model.load_state_dict(torch.load(f'{OUTPUT_DIR}/best_model.pt'))
model.eval()

_, _, predictions, actual_labels = eval_epoch(model, val_loader, DEVICE)

val_data_eval = val_data.copy()
val_data_eval['predicted'] = predictions

print('=' * 50)
print('PER-LANGUAGE EVALUATION RESULTS')
print('=' * 50)

results = {}
for lang in ['fr', 'en', 'es']:
    lang_data = val_data_eval[val_data_eval['language'] == lang]
    if len(lang_data) > 0:
        lang_f1 = f1_score(lang_data['label'], lang_data['predicted'], average='weighted')
        results[lang] = lang_f1
        status = 'PASS' if lang_f1 >= 0.70 else 'FAIL'
        print(f'{lang.upper()} (n={len(lang_data)}): F1 = {lang_f1:.4f} [{status}]')

print()
overall_f1 = f1_score(actual_labels, predictions, average='weighted')
status = 'PASS' if overall_f1 >= 0.70 else 'FAIL'
print(f'OVERALL Val F1: {overall_f1:.4f} [{status}]')

## Save Model & Results

In [ ]:
import json

# Save model
model_path = f'{OUTPUT_DIR}/xlmr_laughter_classifier'
model.save_pretrained(model_path)
tokenizer.save_pretrained(model_path)

# Save history
with open(f'{OUTPUT_DIR}/training_history.json', 'w') as f:
    json.dump(history, f)

# Save results
results_data = {
    'overall_f1': overall_f1,
    'per_language_f1': results,
    'best_val_f1': best_val_f1
}
with open(f'{OUTPUT_DIR}/results.json', 'w') as f:
    json.dump(results_data, f, indent=2)

print('=' * 50)
print('SAVED FILES')
print('=' * 50)
for f in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(f'{OUTPUT_DIR}/{f}') / (1024 * 1024)
    print(f'  {f}: {size:.2f} MB')
print('\nDone!')

## Summary

**Target:** French, English, Spanish F1 >= 0.70

**Output:** `/content/drive/MyDrive/standup4ai_models/`